In [ ]:
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, pipeline
import re
import ast
import pandas as pd


In [ ]:
dataset = load_dataset("openlifescienceai/medmcqa")

In [ ]:
def clean(text):
    return re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL).strip()

In [ ]:
def generate_prompt_for_question(row,
                                 question_col='question',
                                 option_a_col = 'opa',
                                 option_b_col = 'opb',
                                 option_c_col = 'opc',
                                 option_d_col = 'opd',
                                 correct_option = 'cop',
                                 include_options=True,
                                 include_correct_option = False,
                                 context_col=None):
    question_text = row[question_col]
    options = f"a) {row[option_a_col]}\nb) {row[option_b_col]}\nc) {row[option_c_col]}\nd) {row[option_d_col]}"
    correct_option = row[correct_option]
    
    user_prompt_delimiter = "-----\n"
    user_prompt_question = f"Question:\n{question_text}\n"
    user_prompt_options = f"Options:\n{options}\n"

    user_prompt = user_prompt_delimiter + user_prompt_question
    if include_options:
        user_prompt += user_prompt_options
    if include_correct_option:
        correct_option = f"Correct option: {correct_option}\n"
        user_prompt += correct_option

    user_prompt += user_prompt_delimiter
    
    if context_col is not None:
        mcq_context = f"""Context:\n-----\n{row[context_col]}\n-----\n"""
        user_prompt = mcq_context + user_prompt

    return user_prompt

In [ ]:
def answer_mcq_hf(mcq, model_name,tokenizer, temperature):
    prompt = f"""
       You are tasked with answering multiple-choice questions, containing 4 different answer options - a, b, c and d.
       You are given some context to help you answer the question.
       Provide just a single letter corresponding to the correct option as the response.
       For example: "a", "b", "c", or "d"
    """

    
    pipe = pipeline(
        "text-generation",
        model=model_name,
        tokenizer=tokenizer,
        device_map="cuda",
        dtype="bfloat16"
    )
    
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": mcq}
    ]
    
    response = pipe(
        messages,
        max_new_tokens=2048,
        temperature=temperature,
        top_p=1.0,
        do_sample=True,
        return_full_text=False
    )
    return clean(response[0]['generated_text'])[0]

In [ ]:
def answer_mcq(mcq, model_name,tokenizer, temperature):
    ...

In [ ]:
def for_a_model(dataset, model_name, save_name, use_ollama=False):
    if not use_ollama:
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            use_fast=True,
            trust_remote_code=True
        )
    else:
        tokenizer = None
    
    result = pd.DataFrame(columns=["mcq", "cop", "llm_cop"])
    conv = {
        "a": 1,
        "b": 2,
        "c": 3,
        "d": 4
    }
    
    for idx in range(len(dataset)):
        mcq = generate_prompt_for_question(dataset.loc[idx])
        nb_try = 0
        
        while True:
            try:
                generated = (
                    answer_mcq_hf(mcq, model_name, tokenizer, temperature=0.1)
                    if not use_ollama
                    else answer_mcq(mcq, model_name, temperature=0.1)
                )
                generated = conv[generated]
                break
            except (SyntaxError, KeyError):
                print("SyntaxError ou KeyError détectée, relance...")
                nb_try += 1
                if nb_try == 5:
                    print("Nombre d'essai dépassé, passage au Lisa Sheet suivant")
                    generated = None
                    break
        
        if generated is not None:
            result.loc[idx] = [mcq, dataset.loc[idx]["cop"], generated]
    
    result.to_csv(f"../data/answerability/{save_name}.csv", index=False)
    return result

In [ ]:
all_dataset = concatenate_datasets([dataset["train"],dataset["test"],dataset["validation"]])
df = all_dataset.to_pandas()
df_grouped = df.groupby("subject_name")

In [ ]:
dataset_answerability = pd.concat(
    [group.sample(n=min(200, len(group))) for name, group in df_grouped if name != "Unknown"],
    ignore_index=True
)
print(len(dataset_answerability))


In [11]:
model_name = "Qwen/Qwen3-0.6B"
save_name = "qwen3_6b"
result = for_a_model(dataset_answerability,model_name,save_name)

percentage = (result["cop"] == result["llm_cop"]).mean() * 100
print(f"Answerability of {model_name}: {percentage:.2f}%")

KeyboardInterrupt: 

In [ ]:
dataset_answerability = pd.read_json("../data/mcqs_answerability.json")
result = for_a_model(dataset_answerability,model_name,save_name)

percentage = (result["cop"] == result["llm_cop"]).mean() * 100
print(f"Answerability of {model_name}: {percentage:.2f}%")